In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/caoxiyang/damoxing.git

Cloning into 'damoxing'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 134 (delta 14), reused 134 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 731.77 KiB | 18.29 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [6]:
!pip install --upgrade "chronos-forecasting>=2.0"
!pip install "transformers>=4.41,<5"

In [ ]:
!pip install autogluon
!pip install --upgrade pyarrow

In [3]:
# 1. 安装 autogluon
!pip install autogluon

# 2. 强制卸载现有的 pyarrow 并安装特定版本
!pip uninstall -y pyarrow
!pip install --no-cache-dir pyarrow==17.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of opentelemetry-s

In [ ]:
# 1. 强力卸载所有冲突包
# 1. 强力删除系统自带的、可能引起冲突的包
!pip uninstall -y pyarrow datasets autogluon chronos-forecasting

# 2. 安装 Chronos2 核心依赖的“稳定三角”：
# pyarrow 14.0.1 + datasets 2.15.0 + transformers 4.44.2
# 这组版本在 Python 3.12 上避开了大部分 PyExtensionType 的导出问题
!pip install --no-cache-dir pyarrow==14.0.1 datasets==2.15.0 
!pip install --no-cache-dir "transformers>=4.41,<5.0"
!pip install --no-cache-dir chronos-forecasting autogluon

# 3. 强制清理动态库缓存（黑科技）
import os
import signal

# 这一步会尝试杀掉当前的 Python 进程，强迫 Kaggle 重启一个新的、干净的内核
os.kill(os.getpid(), signal.SIGTERM)

Found existing installation: pyarrow 20.0.0
Uninstalling pyarrow-20.0.0:
  Successfully uninstalled pyarrow-20.0.0
Found existing installation: datasets 2.15.0
Uninstalling datasets-2.15.0:
  Successfully uninstalled datasets-2.15.0
Found existing installation: autogluon 1.5.0
Uninstalling autogluon-1.5.0:
  Successfully uninstalled autogluon-1.5.0
Found existing installation: chronos-forecasting 2.2.2
Uninstalling chronos-forecasting-2.2.2:
  Successfully uninstalled chronos-forecasting-2.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 217.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 36.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>

In [9]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

from chronos import Chronos2Pipeline
model_dir = "/root/root/autodl-tmp/demo/chronos-2"
print("exists:", os.path.isdir(model_dir))
print("config:", os.path.isfile(os.path.join(model_dir, "config.json")))
print("weights:", os.path.isfile(os.path.join(model_dir, "model.safetensors")))
# --- 1. Read data ---
file_path = "/kaggle/input/datasets/zhanyi1/dataset/qqqqqqqq20240103_20251102_1315140401.csv"
print(f"Reading: {file_path}")
df = pd.read_csv(file_path)
df["plantid"] = df["plantid"].astype(str)
df["date"] = pd.to_datetime(df["date"])

# sort by id/time
df = df.sort_values(by=["plantid", "date"]).reset_index(drop=True)

# --- 2. Normalize target ---
target_col = "corrected_scada_power"
max_power = 252000
print(f"Max power: {max_power:.2f} (for normalization)")
df[target_col] = df[target_col] / max_power

# --- 3. Split by timesteps (per series length) ---
sample_id = df["plantid"].iloc[0]
single_ts_length = len(df[df["plantid"] == sample_id])

pred_steps = int(single_ts_length * 0.1)   # last 10% for test
val_steps = int(single_ts_length * 0.2)    # middle 20% for val
train_steps = single_ts_length - pred_steps - val_steps

print("--- Per series split ---")
print(f"series length: {single_ts_length}")
print(f"pred_steps (10%): {pred_steps}")
print(f"val_steps (20%): {val_steps}")
print(f"train_steps (70%): {train_steps}")

# split: 0-90% for context, 0-70% for train
def slice_by_steps(group, end_idx):
    # end_idx is exclusive
    return group.iloc[:end_idx]

train_df_list = []
context_df_list = []
test_df_list = []

for item_id, g in df.groupby("plantid"):
    g = g.sort_values("date")
    g_train = slice_by_steps(g, train_steps)
    g_context = slice_by_steps(g, train_steps + val_steps)
    g_test = g.iloc[-pred_steps:]

    train_df_list.append(g_train)
    context_df_list.append(g_context)
    test_df_list.append(g_test)

train_df = pd.concat(train_df_list).reset_index(drop=True)
context_df = pd.concat(context_df_list).reset_index(drop=True)
test_df = pd.concat(test_df_list).reset_index(drop=True)

# --- 4. Build fine-tuning inputs (no covariates) ---
train_inputs = []
for item_id, g in train_df.groupby("plantid"):
    train_inputs.append({
        "target": g[target_col].values
    })

# --- 5. Load model and fine-tune ---
# choose device_map="cuda" if you have GPU
# --- 5. Load model and fine-tune ---
import torch, gc

# 1. 强力清理显存残留
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_gpu()

# 2. 加载模型：关键是 device_map="auto"
# 这会自动把模型权重分布在两张 GPU 上，单张卡就不会那么挤了
pipeline = Chronos2Pipeline.from_pretrained(
    r"/kaggle/working/damoxing/chronos-2",
    device_map="auto", 
    torch_dtype=torch.bfloat16, # 使用 bfloat16 节省一半显存 (T4 支持)
    local_files_only=True
)

# 3. 开始微调：大幅下调 batch_size
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=24,      # 微调时的预测长度建议设短（如24或48）
    context_length=512,        # 必须限制上下文长度，否则默认值可能很大导致 OOM
    finetune_mode="lora",      # 强烈建议用 lora，full 模式在 T4 上极难跑通
    batch_size=1,              # 降到最低 1，如果还崩就说明单条数据太长
    gradient_accumulation_steps=16, # 通过积累 16 步来达到等效 batch_size=16
    learning_rate=1e-4,
    num_steps=500,             # 步数可以先设少点测试
    logging_steps=50,
    # 启用以下设置防止碎片化
    tf32=False,                # T4 不支持 tf32，关闭以防万一
)
# --- 6. Predict ---
# 建议：将 prediction_length 限制在 1024 以内（例如 512 或 720）
actual_pred_steps = min(pred_steps, 512) 

print(f"正在进行推理，预测步数设定为: {actual_pred_steps}")

pred_df = finetuned_pipeline.predict_df(
    context_df,
    prediction_length=actual_pred_steps, # 缩短预测长度
    context_length=1024,                # 显式限制模型只看最近的 1024 个历史点
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="plantid",
    timestamp_column="date",
    target=target_col,
    batch_size=8,                       # 减小预测时的 batch_size
)
# --- 7. Evaluate and plot (first id) ---
item_id = df["plantid"].iloc[0]

# pred_df is usually a flat frame with id_column; filter by id instead of .loc
pred_one = pred_df[pred_df["plantid"] == item_id].set_index("date").sort_index()
if pred_one.empty:
    raise KeyError(f"No predictions for plantid={item_id}")

pred_mean = pred_one["0.5"] * max_power
pred_01 = pred_one["0.1"] * max_power
pred_09 = pred_one["0.9"] * max_power

y_true_ts = test_df[test_df["plantid"] == item_id].set_index("date")[target_col] * max_power

common_index = y_true_ts.index.intersection(pred_mean.index)
y_true_aligned = y_true_ts.loc[common_index]
y_pred_aligned = pred_mean.loc[common_index]

if len(y_true_aligned) == 0:
    print("Error: no overlapping timestamps for evaluation.")
else:
    mae = mean_absolute_error(y_true_aligned, y_pred_aligned)
    rmse = np.sqrt(mean_squared_error(y_true_aligned, y_pred_aligned))
    accuracy = (1 - (mae / max_power)) * 100

    print(f"--- Eval (ID: {item_id}) ---")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")

    plt.figure(figsize=(12, 6))
    plt.plot(y_true_ts.index, y_true_ts.values, label="Ground Truth", color="black", linewidth=1.5)
    plt.plot(pred_mean.index, pred_mean.values, label="Prediction (median)", color="#FF3333", linewidth=1.5, linestyle="--")
    plt.fill_between(pred_mean.index, pred_01, pred_09, color="#FF3333", alpha=0.15, label="CI (0.1-0.9)")

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H"))
    plt.gcf().autofmt_xdate()
    plt.title(f"Forecast for {item_id} (Acc: {accuracy:.2f}%)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.show()


ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

from chronos import Chronos2Pipeline
model_dir = "/root/root/autodl-tmp/demo/chronos-2"
print("exists:", os.path.isdir(model_dir))
print("config:", os.path.isfile(os.path.join(model_dir, "config.json")))
print("weights:", os.path.isfile(os.path.join(model_dir, "model.safetensors")))
# --- 1. Read data ---
file_path = "/kaggle/input/datasets/zhanyi1/dataset/qqqqqqqq20240103_20251102_1315140401.csv"
print(f"Reading: {file_path}")
df = pd.read_csv(file_path)
df["plantid"] = df["plantid"].astype(str)
df["date"] = pd.to_datetime(df["date"])

# sort by id/time
df = df.sort_values(by=["plantid", "date"]).reset_index(drop=True)

# --- 2. Normalize target ---
target_col = "corrected_scada_power"
max_power = 252000
print(f"Max power: {max_power:.2f} (for normalization)")
df[target_col] = df[target_col] / max_power

# --- 3. Split by timesteps (per series length) ---
sample_id = df["plantid"].iloc[0]
single_ts_length = len(df[df["plantid"] == sample_id])

pred_steps = int(single_ts_length * 0.1)   # last 10% for test
val_steps = int(single_ts_length * 0.2)    # middle 20% for val
train_steps = single_ts_length - pred_steps - val_steps

print("--- Per series split ---")
print(f"series length: {single_ts_length}")
print(f"pred_steps (10%): {pred_steps}")
print(f"val_steps (20%): {val_steps}")
print(f"train_steps (70%): {train_steps}")

# split: 0-90% for context, 0-70% for train
def slice_by_steps(group, end_idx):
    # end_idx is exclusive
    return group.iloc[:end_idx]

train_df_list = []
context_df_list = []
test_df_list = []

for item_id, g in df.groupby("plantid"):
    g = g.sort_values("date")
    g_train = slice_by_steps(g, train_steps)
    g_context = slice_by_steps(g, train_steps + val_steps)
    g_test = g.iloc[-pred_steps:]

    train_df_list.append(g_train)
    context_df_list.append(g_context)
    test_df_list.append(g_test)

train_df = pd.concat(train_df_list).reset_index(drop=True)
context_df = pd.concat(context_df_list).reset_index(drop=True)
test_df = pd.concat(test_df_list).reset_index(drop=True)

# --- 4. Build fine-tuning inputs (no covariates) ---
train_inputs = []
for item_id, g in train_df.groupby("plantid"):
    train_inputs.append({
        "target": g[target_col].values
    })

# --- 5. Load model and fine-tune ---
# choose device_map="cuda" if you have GPU
# --- 5. Load model and fine-tune ---
import torch, gc

# 1. 强力清理显存残留
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_gpu()

# 2. 加载模型：关键是 device_map="auto"
# 这会自动把模型权重分布在两张 GPU 上，单张卡就不会那么挤了
pipeline = Chronos2Pipeline.from_pretrained(
    r"/kaggle/working/damoxing/chronos-2",
    device_map="auto", 
    torch_dtype=torch.bfloat16, # 使用 bfloat16 节省一半显存 (T4 支持)
    local_files_only=True
)

# 3. 开始微调：大幅下调 batch_size
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=24,      # 微调时的预测长度建议设短（如24或48）
    context_length=512,        # 必须限制上下文长度，否则默认值可能很大导致 OOM
    finetune_mode="lora",      # 强烈建议用 lora，full 模式在 T4 上极难跑通
    batch_size=4,              # 降到最低 1，如果还崩就说明单条数据太长
    gradient_accumulation_steps=16, # 通过积累 16 步来达到等效 batch_size=16
    learning_rate=1e-4,
    num_steps=1000,             # 步数可以先设少点测试
    logging_steps=50,
    # 启用以下设置防止碎片化
    tf32=False,                # T4 不支持 tf32，关闭以防万一
)
# --- 6. Predict ---
# 建议：将 prediction_length 限制在 1024 以内（例如 512 或 720）
actual_pred_steps = min(pred_steps, 512) 

print(f"正在进行推理，预测步数设定为: {actual_pred_steps}")

pred_df = finetuned_pipeline.predict_df(
    context_df,
    prediction_length=actual_pred_steps, # 缩短预测长度
    context_length=1024,                # 显式限制模型只看最近的 1024 个历史点
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="plantid",
    timestamp_column="date",
    target=target_col, 
    batch_size=16,                       # 减小预测时的 batch_size
)
# --- 7. Evaluate and plot (first id) ---
item_id = df["plantid"].iloc[0]

# pred_df is usually a flat frame with id_column; filter by id instead of .loc
pred_one = pred_df[pred_df["plantid"] == item_id].set_index("date").sort_index()
if pred_one.empty:
    raise KeyError(f"No predictions for plantid={item_id}")

pred_mean = pred_one["0.5"] * max_power
pred_01 = pred_one["0.1"] * max_power
pred_09 = pred_one["0.9"] * max_power

y_true_ts = test_df[test_df["plantid"] == item_id].set_index("date")[target_col] * max_power

common_index = y_true_ts.index.intersection(pred_mean.index)
y_true_aligned = y_true_ts.loc[common_index]
y_pred_aligned = pred_mean.loc[common_index]

if len(y_true_aligned) == 0:
    print("Error: no overlapping timestamps for evaluation.")
else:
    mae = mean_absolute_error(y_true_aligned, y_pred_aligned)
    rmse = np.sqrt(mean_squared_error(y_true_aligned, y_pred_aligned))
    accuracy = (1 - (mae / max_power)) * 100

    print(f"--- Eval (ID: {item_id}) ---")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")

    plt.figure(figsize=(12, 6))
    plt.plot(y_true_ts.index, y_true_ts.values, label="Ground Truth", color="black", linewidth=1.5)
    plt.plot(pred_mean.index, pred_mean.values, label="Prediction (median)", color="#FF3333", linewidth=1.5, linestyle="--")
    plt.fill_between(pred_mean.index, pred_01, pred_09, color="#FF3333", alpha=0.15, label="CI (0.1-0.9)")

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H"))
    plt.gcf().autofmt_xdate()
    plt.title(f"Forecast for {item_id} (Acc: {accuracy:.2f}%)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.show()


In [5]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

from chronos import Chronos2Pipeline
model_dir = "/root/root/autodl-tmp/demo/chronos-2"
print("exists:", os.path.isdir(model_dir))
print("config:", os.path.isfile(os.path.join(model_dir, "config.json")))
print("weights:", os.path.isfile(os.path.join(model_dir, "model.safetensors")))
# --- 1. Read data ---
file_path = "/kaggle/input/datasets/zhanyi1/dataset/qqqqqqqq20240103_20251102_1315140401.csv"
print(f"Reading: {file_path}")
df = pd.read_csv(file_path)
df["plantid"] = df["plantid"].astype(str)
df["date"] = pd.to_datetime(df["date"])

# sort by id/time
df = df.sort_values(by=["plantid", "date"]).reset_index(drop=True)

# --- 2. Normalize target ---
target_col = "corrected_scada_power"
max_power = 252000
print(f"Max power: {max_power:.2f} (for normalization)")
df[target_col] = df[target_col] / max_power

# --- 3. Split by timesteps (per series length) ---
sample_id = df["plantid"].iloc[0]
single_ts_length = len(df[df["plantid"] == sample_id])

pred_steps = int(single_ts_length * 0.1)   # last 10% for test
val_steps = int(single_ts_length * 0.2)    # middle 20% for val
train_steps = single_ts_length - pred_steps - val_steps

print("--- Per series split ---")
print(f"series length: {single_ts_length}")
print(f"pred_steps (10%): {pred_steps}")
print(f"val_steps (20%): {val_steps}")
print(f"train_steps (70%): {train_steps}")

# split: 0-90% for context, 0-70% for train
def slice_by_steps(group, end_idx):
    # end_idx is exclusive
    return group.iloc[:end_idx]

train_df_list = []
context_df_list = []
test_df_list = []

for item_id, g in df.groupby("plantid"):
    g = g.sort_values("date")
    g_train = slice_by_steps(g, train_steps)
    g_context = slice_by_steps(g, train_steps + val_steps)
    g_test = g.iloc[-pred_steps:]

    train_df_list.append(g_train)
    context_df_list.append(g_context)
    test_df_list.append(g_test)

train_df = pd.concat(train_df_list).reset_index(drop=True)
context_df = pd.concat(context_df_list).reset_index(drop=True)
test_df = pd.concat(test_df_list).reset_index(drop=True)

# --- 4. Build fine-tuning inputs (no covariates) ---
train_inputs = []
for item_id, g in train_df.groupby("plantid"):
    train_inputs.append({
        "target": g[target_col].values
    })

# --- 5. Load model and fine-tune ---
# choose device_map="cuda" if you have GPU
# --- 5. Load model and fine-tune ---
import torch, gc

# 1. 强力清理显存残留
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_gpu()

# 2. 加载模型：关键是 device_map="auto"
# 这会自动把模型权重分布在两张 GPU 上，单张卡就不会那么挤了
pipeline = Chronos2Pipeline.from_pretrained(
    r"/kaggle/working/damoxing/chronos-2",
    device_map="auto", 
    torch_dtype=torch.bfloat16, # 使用 bfloat16 节省一半显存 (T4 支持)
    local_files_only=True
)

# 3. 开始微调：大幅下调 batch_size
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=24,      # 微调时的预测长度建议设短（如24或48）
    context_length=512,        # 必须限制上下文长度，否则默认值可能很大导致 OOM
    finetune_mode="lora",      # 强烈建议用 lora，full 模式在 T4 上极难跑通
    batch_size=8,              # 降到最低 1，如果还崩就说明单条数据太长
    gradient_accumulation_steps=16, # 通过积累 16 步来达到等效 batch_size=16
    learning_rate=1e-4,
    num_steps=500,             # 步数可以先设少点测试
    logging_steps=50,
    # 启用以下设置防止碎片化
    tf32=False,                # T4 不支持 tf32，关闭以防万一
)
# --- 6. Predict ---
# 建议：将 prediction_length 限制在 1024 以内（例如 512 或 720）
actual_pred_steps = min(pred_steps, 512) 

print(f"正在进行推理，预测步数设定为: {actual_pred_steps}")

pred_df = finetuned_pipeline.predict_df(
    context_df,
    prediction_length=actual_pred_steps, # 缩短预测长度
    context_length=1024,                # 显式限制模型只看最近的 1024 个历史点
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="plantid",
    timestamp_column="date",
    target=target_col,
    batch_size=16, 
   
)
# --- 7. Evaluate and plot (first id) ---
item_id = df["plantid"].iloc[0]

# pred_df is usually a flat frame with id_column; filter by id instead of .loc
pred_one = pred_df[pred_df["plantid"] == item_id].set_index("date").sort_index()
if pred_one.empty:
    raise KeyError(f"No predictions for plantid={item_id}")

pred_mean = pred_one["0.5"] * max_power
pred_01 = pred_one["0.1"] * max_power
pred_09 = pred_one["0.9"] * max_power

y_true_ts = test_df[test_df["plantid"] == item_id].set_index("date")[target_col] * max_power

common_index = y_true_ts.index.intersection(pred_mean.index)
y_true_aligned = y_true_ts.loc[common_index]
y_pred_aligned = pred_mean.loc[common_index]

if len(y_true_aligned) == 0:
    print("Error: no overlapping timestamps for evaluation.")
else:
    mae = mean_absolute_error(y_true_aligned, y_pred_aligned)
    rmse = np.sqrt(mean_squared_error(y_true_aligned, y_pred_aligned))
    accuracy = (1 - (mae / max_power)) * 100

    print(f"--- Eval (ID: {item_id}) ---")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")

    plt.figure(figsize=(12, 6))
    plt.plot(y_true_ts.index, y_true_ts.values, label="Ground Truth", color="black", linewidth=1.5)
    plt.plot(pred_mean.index, pred_mean.values, label="Prediction (median)", color="#FF3333", linewidth=1.5, linestyle="--")
    plt.fill_between(pred_mean.index, pred_01, pred_09, color="#FF3333", alpha=0.15, label="CI (0.1-0.9)")

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H"))
    plt.gcf().autofmt_xdate()
    plt.title(f"Forecast for {item_id} (Acc: {accuracy:.2f}%)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.show()


ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [1]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler  # <--- 新增：引入标准化工具
import torch
import gc
from chronos import Chronos2Pipeline

# ==========================================
# 0. 清理显存 (防止之前的报错残留)
# ==========================================
gc.collect()
torch.cuda.empty_cache()

model_dir = "/kaggle/working/damoxing/chronos-2"
print("exists:", os.path.isdir(model_dir))

# ==========================================
# --- 1. 读取数据 ---
# ==========================================
file_path = "/kaggle/input/datasets/zhanyi1/dataset/qqqqqqqq20240103_20251102_1315140401.csv"
print(f"Reading: {file_path}")
df = pd.read_csv(file_path)
df["plantid"] = df["plantid"].astype(str)
df["date"] = pd.to_datetime(df["date"])

# 排序
df = df.sort_values(by=["plantid", "date"]).reset_index(drop=True)

# ==========================================
# --- 2. 目标值 Z-score 标准化 (核心优化) ---
# ==========================================
target_col = "corrected_scada_power"
scaler = StandardScaler()  # 实例化标准化器

# 拟合并且转换数据
print("正在使用 StandardScaler 进行数据标准化...")
df[target_col] = scaler.fit_transform(df[[target_col]])

# ==========================================
# --- 3. 划分数据集 ---
# ==========================================
sample_id = df["plantid"].iloc[0]
single_ts_length = len(df[df["plantid"] == sample_id])

pred_steps = int(single_ts_length * 0.1)   # 测试集长度
val_steps = int(single_ts_length * 0.2)    # 验证集长度
train_steps = single_ts_length - pred_steps - val_steps

print("--- Per series split ---")
print(f"series length: {single_ts_length}")
print(f"train_steps (70%): {train_steps}")

def slice_by_steps(group, end_idx):
    return group.iloc[:end_idx]

train_df_list, context_df_list, test_df_list = [], [], []

for item_id, g in df.groupby("plantid"):
    g = g.sort_values("date")
    g_train = slice_by_steps(g, train_steps)
    g_context = slice_by_steps(g, train_steps + val_steps)
    g_test = g.iloc[-pred_steps:]

    train_df_list.append(g_train)
    context_df_list.append(g_context)
    test_df_list.append(g_test)

train_df = pd.concat(train_df_list).reset_index(drop=True)
context_df = pd.concat(context_df_list).reset_index(drop=True)
test_df = pd.concat(test_df_list).reset_index(drop=True)

# --- 4. 构建训练输入 ---
train_inputs = []
for item_id, g in train_df.groupby("plantid"):
    train_inputs.append({
        "target": g[target_col].values
    })

# ==========================================
# --- 5. 加载模型并微调 ---
# ==========================================
pipeline = Chronos2Pipeline.from_pretrained(
    model_dir,
    device_map="auto",  # 自动分配到两张 GPU
    local_files_only=True
)

print("开始模型微调...")
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=48,       # 微调时的内部预测长度
    context_length=512,         # 限制历史窗口，防止 OOM
    finetune_mode="lora",       # LoRA 微调
    learning_rate=1e-4,
    num_steps=500,              # 跑 500 步
    batch_size=4,               # 控制单卡显存
    gradient_accumulation_steps=8,
    logging_steps=50,
)

# ==========================================
# --- 6. 预测 (移除 0.95 分位数) ---
# ==========================================
actual_pred_steps = min(pred_steps, 512) 
print(f"\n正在进行推理，预测步数设定为: {actual_pred_steps} ...")

pred_df = finetuned_pipeline.predict_df(
    context_df,
    prediction_length=actual_pred_steps,
    quantile_levels=[0.1, 0.5, 0.9], # <--- 移除了 0.95
    id_column="plantid",
    timestamp_column="date",
    target=target_col,
    batch_size=4,  
)

# ==========================================
# --- 7. 反标准化、评估与画图 ---
# ==========================================
item_id = df["plantid"].iloc[0]
pred_one = pred_df[pred_df["plantid"] == item_id].set_index("date").sort_index()

# 反标准化
pred_mean_vals = scaler.inverse_transform(pred_one[["0.5"]]).flatten()
pred_01_vals = scaler.inverse_transform(pred_one[["0.1"]]).flatten()
pred_09_vals = scaler.inverse_transform(pred_one[["0.9"]]).flatten()

pred_mean = pd.Series(pred_mean_vals, index=pred_one.index)
pred_01 = pd.Series(pred_01_vals, index=pred_one.index)
pred_09 = pd.Series(pred_09_vals, index=pred_one.index)

# 真实值反标准化
test_one = test_df[test_df["plantid"] == item_id].set_index("date").iloc[:actual_pred_steps]
y_true_ts_vals = scaler.inverse_transform(test_one[[target_col]]).flatten()
y_true_ts = pd.Series(y_true_ts_vals, index=test_one.index)

# 对齐并计算误差
common_index = y_true_ts.index.intersection(pred_mean.index)
y_true_aligned = y_true_ts.loc[common_index]
y_pred_aligned = pred_mean.loc[common_index]

mae = mean_absolute_error(y_true_aligned, y_pred_aligned)
rmse = np.sqrt(mean_squared_error(y_true_aligned, y_pred_aligned))

# 计算准确率 (沿用你最初的公式)
max_power = 252000.00 
accuracy = (1 - (mae / max_power)) * 100

print(f"--- Eval (ID: {item_id}) ---")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%") # <--- 打印准确率

# 画图
plt.figure(figsize=(14, 7))
plt.plot(y_true_ts.index, y_true_ts.values, label="Ground Truth", color="black", linewidth=1.5)
plt.plot(pred_mean.index, pred_mean.values, label="Prediction (Median)", color="#FF3333", linewidth=1.5, linestyle="-")
plt.fill_between(pred_mean.index, pred_01, pred_09, color="#FF3333", alpha=0.15, label="Confidence Interval (10%-90%)")

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.gcf().autofmt_xdate()
plt.title(f"Forecast for {item_id} (Acc: {accuracy:.2f}%)") # <--- 标题显示准确率
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

2026-03-23 13:35:18.146795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774272918.171193     432 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774272918.178957     432 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774272918.199482     432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774272918.199506     432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774272918.199508     432 computation_placer.cc:177] computation placer alr

exists: True
Reading: /kaggle/input/datasets/zhanyi1/dataset/qqqqqqqq20240103_20251102_1315140401.csv
正在使用 StandardScaler 进行数据标准化...
--- Per series split ---
series length: 15930
train_steps (70%): 11151


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['quantiles']
  warnings.warn(


开始模型微调...


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


KeyboardInterrupt: 

In [3]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=== 第一阶段：特征工程 (Feature Engineering) ===")
# 1. 为原始数据集提取时间特征（因为你目前没有天气数据，时间特征是对光伏最重要的）
df_lgb = df.copy()
df_lgb['hour'] = df_lgb['date'].dt.hour
df_lgb['month'] = df_lgb['date'].dt.month
df_lgb['dayofweek'] = df_lgb['date'].dt.dayofweek
# 光伏极其依赖日照，制造一个简单的“是否是白天”特征 (假设早6点到晚19点有太阳)
df_lgb['is_daytime'] = df_lgb['hour'].apply(lambda x: 1 if 6 <= x <= 19 else 0)

# 我们只取你当前测试的那个电站 (item_id) 进行演示
df_item = df_lgb[df_lgb['plantid'] == item_id].sort_values("date").reset_index(drop=True)

print("=== 第二阶段：获取 Chronos 在验证集上的预测，并计算残差 ===")
# 提取用于 Chronos 预测的上下文 (前 70%) 和需要预测的验证集 (中间 20%)
# 注意：之前的代码里 train_steps 是 70%，val_steps 是 20%
train_chunk = df_item.iloc[:train_steps]
val_chunk = df_item.iloc[train_steps : train_steps + val_steps]

# 限制一下验证集的预测长度防止 OOM (比如只取前 512 步来训练 LightGBM)
val_pred_len = min(val_steps, 512)
val_chunk = val_chunk.iloc[:val_pred_len]

# 让 Chronos 预测这部分验证集
print(f"Chronos 正在回测验证集 (长度: {val_pred_len})...")
val_pred_df = finetuned_pipeline.predict_df(
    train_chunk, # 喂给它 70% 的历史
    prediction_length=val_pred_len, # 预测接下来的 20%
    quantile_levels=[0.5], 
    id_column="plantid", timestamp_column="date", target=target_col, batch_size=4
)

# 反标准化：把验证集的真实值和预测值都还原成实际功率
val_pred_median = scaler.inverse_transform(val_pred_df[["0.5"]]).flatten()
val_true_power = scaler.inverse_transform(val_chunk[[target_col]]).flatten()

# 【核心】计算残差 (真实功率 - Chronos预测的功率)
# 晴天中午时，这个值会很大（因为 Chronos 削平了波峰），这就是 LGBM 要学的东西
val_residuals = val_true_power - val_pred_median 

print("=== 第三阶段：训练 LightGBM 专门拟合残差 ===")
features = ['hour', 'month', 'dayofweek', 'is_daytime']

X_train_lgb = val_chunk[features]
y_train_lgb = val_residuals

# 构建 LightGBM 数据集
lgb_train = lgb.Dataset(X_train_lgb, label=y_train_lgb)

# 设置 LGBM 参数 (树模型对极端值敏感，不用太深)
params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'max_depth': 5,
    'num_leaves': 31,
    'verbose': -1,
    'random_state': 42
}

print("LightGBM 正在学习如何修复波峰...")
gbm = lgb.train(params, lgb_train, num_boost_round=150)

print("=== 第四阶段：未来预测大融合 (Test Set) ===")
# 1. 之前你已经用 Chronos 预测了测试集，结果存在 pred_mean 里面 (已经反标准化了)
# pred_mean 就是我们要的 Baseline

# 2. 提取测试集的特征，让 LightGBM 预测误差
test_chunk = test_df[test_df['plantid'] == item_id].iloc[:actual_pred_steps]
X_test_lgb = test_chunk[features]

# LightGBM 预测出来的“需要补偿的波峰高度”
predicted_residuals = gbm.predict(X_test_lgb)

# 3. 【终极融合】: 最终预测 = Chronos基线 + LGBM预测的误差补偿
final_hybrid_predictions = pred_mean.values + predicted_residuals

# 防止出现负数发电量
final_hybrid_predictions = np.clip(final_hybrid_predictions, a_min=0, a_max=None)

print("=== 第五阶段：评估与绘图对比 ===")
# 计算融合后的误差
hybrid_mae = mean_absolute_error(y_true_ts.values, final_hybrid_predictions)
hybrid_accuracy = (1 - (hybrid_mae / max_power)) * 100

print(f"单独 Chronos Accuracy: {accuracy:.2f}%")
print(f"混合模型 (Chronos+LGBM) Accuracy: {hybrid_accuracy:.2f}%")

# 画图：对比单纯 Chronos 和 混合模型
plt.figure(figsize=(14, 7))
plt.plot(y_true_ts.index, y_true_ts.values, label="Ground Truth (真实值)", color="black", linewidth=1.5)

# 画出单纯的 Chronos 预测 (之前被削平的红线)
plt.plot(pred_mean.index, pred_mean.values, label="Chronos Only", color="#FF3333", linestyle="--", alpha=0.7)

# 画出 LGBM 修复后的预测 (看波峰是不是顶上去了！)
plt.plot(pred_mean.index, final_hybrid_predictions, label="Hybrid (Chronos + LGBM) - 重点看波峰", color="blue", linewidth=1.8)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.gcf().autofmt_xdate()
plt.title(f"Hybrid Forecast Comparison (Hybrid Acc: {hybrid_accuracy:.2f}%)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

=== 第一阶段：特征工程 (Feature Engineering) ===


NameError: name 'df' is not defined